# Build Content Guard Android APK (free Google Colab)

This notebook builds a **debug APK** from the Content Guard Flask dashboard using Buildozer, Kivy, and Android System WebView. Colab supplies a temporary Linux build machine; the first Android SDK/NDK build can take 20–60 minutes and may use several GB of disk. Free runtimes can disconnect or reclaim resources, so rerun the build cell if that happens.

## Before running

1. In this project on Windows, run `prepare_android_source.bat`.
2. Open this notebook in Google Colab (`https://colab.research.google.com/`) and choose **Runtime → Change runtime type → CPU** (the free CPU runtime is enough).
3. Run each cell in order. When asked to upload a file, select `dist/ContentGuard-Android-Source.zip`.
4. At the end, download `ContentGuard-Android-1.0.0-debug.apk` and install it on a test device. Allow sideloading for the app used to open the APK if Android asks.

The generated APK is unsigned for release and is for development/testing. It is not a device-wide filter or Android VPN service. Keep the build private if it contains custom/private project code; the uploaded source is processed in your Colab runtime.

In [ ]:
# Install the Android build toolchain packages in Colab's temporary Linux VM.
!apt-get update -qq
!apt-get install -y -qq git zip unzip openjdk-17-jdk autoconf libtool pkg-config zlib1g-dev libncurses-dev cmake libffi-dev libssl-dev libsqlite3-dev libltdl-dev
!java -version
!python3 --version

In [ ]:
# Keep Python build tools isolated from Colab's system Python packages.
!python3 -m venv /content/content-guard-build-venv
!/content/content-guard-build-venv/bin/python -m pip install --upgrade pip setuptools wheel buildozer "Cython<4"
!/content/content-guard-build-venv/bin/buildozer --version

In [ ]:
# Upload the source-only archive produced by prepare_android_source.bat.
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import shutil

uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
if len(archives) != 1:
    raise RuntimeError("Upload exactly one ZIP: ContentGuard-Android-Source.zip")

project_dir = Path("/content/content-guard-android")
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True)
with ZipFile(archives[0]) as source_archive:
    for entry in source_archive.infolist():
        destination = (project_dir / entry.filename).resolve()
        if project_dir.resolve() not in destination.parents and destination != project_dir.resolve():
            raise RuntimeError(f"Unsafe path in source archive: {entry.filename}")
    source_archive.extractall(project_dir)

required = ["app.py", "main.py", "buildozer.spec", "templates", "static"]
missing = [name for name in required if not (project_dir / name).exists()]
if missing:
    raise FileNotFoundError(f"Archive is missing required Android project files: {missing}")

# Colab runs as root; disable Buildozer's interactive root-user warning for this disposable VM.
spec_path = project_dir / "buildozer.spec"
spec_text = spec_path.read_text(encoding="utf-8")
if "warn_on_root = 1" in spec_text:
    spec_path.write_text(spec_text.replace("warn_on_root = 1", "warn_on_root = 0"), encoding="utf-8")
print(f"Project extracted to {project_dir}")
print(f"Buildozer spec: {project_dir / 'buildozer.spec'}")

In [ ]:
# Build the APK. The first run downloads Android SDK/NDK and can take a while.
%cd /content/content-guard-android
!/content/content-guard-build-venv/bin/buildozer -v android debug

In [ ]:
# Copy the generated APK to a predictable name and download it from Colab.
from google.colab import files
from pathlib import Path
import shutil

apk_files = sorted(Path("bin").glob("*.apk"), key=lambda path: path.stat().st_mtime, reverse=True)
if not apk_files:
    raise FileNotFoundError("No APK was produced. Review the Buildozer output above and rerun the build cell.")
download_path = Path("/content/ContentGuard-Android-1.0.0-debug.apk")
shutil.copy2(apk_files[0], download_path)
print(f"Built from: {apk_files[0]}")
print(f"APK size: {download_path.stat().st_size:,} bytes")
files.download(str(download_path))